# 21 — AI Models: Architecture, Training & Selection

**Author:** Bency (Benjamin Adjei)  
**Role:** AI Evaluator | M.S. Cybersecurity, UMBC  
**Certifications:** CompTIA AI+, CISA

---

## Objectives
- Understand the major categories of AI/ML models and their architectures
- Compare supervised, unsupervised, and reinforcement learning paradigms
- Survey foundational model families: CNNs, RNNs, Transformers, Diffusion models
- Understand LLM architecture — tokenisation, attention, context windows
- Evaluate and select the right model for a given task
- Understand fine-tuning, RAG, and prompt engineering as adaptation strategies
- Score and compare models on capability, cost, and safety dimensions

## 1. Setup & Imports

In [ ]:
import json
import math
from datetime import datetime, timezone
from collections import Counter

## 2. AI/ML Learning Paradigms

```
  ARTIFICIAL INTELLIGENCE
  └── Machine Learning
        ├── Supervised Learning      — labelled data → predict outcome
        ├── Unsupervised Learning    — unlabelled data → discover structure
        ├── Semi-Supervised Learning — small labelled + large unlabelled
        ├── Reinforcement Learning   — agent + environment + reward signal
        └── Self-Supervised Learning — labels derived from data itself (GPT, BERT)
  └── Deep Learning
        ├── CNNs  — images, spatial patterns
        ├── RNNs / LSTMs — sequences, time series
        ├── Transformers — language, code, multimodal
        └── Diffusion Models — image/audio generation
  └── Generative AI
        ├── LLMs (GPT, Claude, Gemini)
        ├── Image generation (Stable Diffusion, DALL-E)
        ├── Code generation (Copilot, CodeLlama)
        └── Multimodal (GPT-4o, Gemini Ultra)
```

## 3. Model Categories & Use Cases

In [ ]:
MODEL_CATEGORIES = [
    {
        'category'    : 'Large Language Models (LLMs)',
        'architecture': 'Transformer (decoder-only)',
        'task'        : 'Text generation, Q&A, summarisation, code, reasoning',
        'examples'    : ['GPT-4o (OpenAI)', 'Claude 3.5 Sonnet (Anthropic)', 'Gemini 1.5 Pro (Google)', 'Llama 3.1 (Meta)'],
        'input'       : 'Text / tokens',
        'output'      : 'Text / tokens',
        'key_metric'  : 'Context window size, MMLU score, hallucination rate'
    },
    {
        'category'    : 'Embedding Models',
        'architecture': 'Transformer (encoder-only)',
        'task'        : 'Semantic search, similarity, clustering, RAG retrieval',
        'examples'    : ['text-embedding-3-large (OpenAI)', 'bge-large (BAAI)', 'Cohere Embed v3'],
        'input'       : 'Text',
        'output'      : 'Dense vector (embedding)',
        'key_metric'  : 'MTEB benchmark score, embedding dimensions'
    },
    {
        'category'    : 'Vision Models (CNNs / ViT)',
        'architecture': 'Convolutional Neural Network / Vision Transformer',
        'task'        : 'Image classification, object detection, segmentation',
        'examples'    : ['ResNet-50', 'EfficientNet', 'YOLO v8', 'ViT (Google)', 'CLIP (OpenAI)'],
        'input'       : 'Images / video frames',
        'output'      : 'Class labels, bounding boxes, masks',
        'key_metric'  : 'Top-1 accuracy (ImageNet), mAP, FPS'
    },
    {
        'category'    : 'Multimodal Models',
        'architecture': 'Transformer + vision encoder fusion',
        'task'        : 'Image + text understanding, visual Q&A, document analysis',
        'examples'    : ['GPT-4o', 'Gemini 1.5 Pro', 'Claude 3.5 Sonnet', 'LLaVA'],
        'input'       : 'Text + Images + Audio + Video',
        'output'      : 'Text / structured data',
        'key_metric'  : 'MMMU score, DocVQA, visual reasoning benchmarks'
    },
    {
        'category'    : 'Code Generation Models',
        'architecture': 'Transformer (decoder, code-trained)',
        'task'        : 'Code completion, generation, debugging, refactoring',
        'examples'    : ['GitHub Copilot (GPT-4o)', 'CodeLlama 70B', 'DeepSeek Coder', 'Claude 3.5 Sonnet'],
        'input'       : 'Code + natural language',
        'output'      : 'Code',
        'key_metric'  : 'HumanEval pass@1, SWE-bench'
    },
    {
        'category'    : 'Image Generation Models',
        'architecture': 'Diffusion / GAN / Flow-based',
        'task'        : 'Text-to-image, image editing, style transfer',
        'examples'    : ['DALL-E 3 (OpenAI)', 'Stable Diffusion 3', 'Midjourney v6', 'Imagen 3 (Google)'],
        'input'       : 'Text prompt (+ optional reference image)',
        'output'      : 'Images',
        'key_metric'  : 'FID score, CLIP score, human preference rating'
    },
    {
        'category'    : 'Time Series / Forecasting Models',
        'architecture': 'LSTM, Transformer, N-BEATS',
        'task'        : 'Demand forecasting, anomaly detection, financial prediction',
        'examples'    : ['TimeGPT', 'Chronos (Amazon)', 'Prophet (Meta)', 'LSTM networks'],
        'input'       : 'Numerical time series',
        'output'      : 'Future values / anomaly scores',
        'key_metric'  : 'MAE, RMSE, MAPE'
    },
    {
        'category'    : 'Tabular / Classical ML Models',
        'architecture': 'Tree-based, linear, ensemble',
        'task'        : 'Classification, regression, fraud detection, risk scoring',
        'examples'    : ['XGBoost', 'LightGBM', 'Random Forest', 'Logistic Regression'],
        'input'       : 'Structured tabular data',
        'output'      : 'Labels, probabilities, continuous values',
        'key_metric'  : 'AUC-ROC, F1 score, accuracy'
    },
]

print('=== AI MODEL CATEGORIES ===\n')
for m in MODEL_CATEGORIES:
    print(f"  [{m['category']}]")
    print(f"    Architecture : {m['architecture']}")
    print(f"    Task         : {m['task'][:70]}")
    print(f"    Examples     : {', '.join(m['examples'][:3])}")
    print(f"    Key metric   : {m['key_metric']}\n")

## 4. LLM Architecture Deep Dive

```
  INPUT TEXT
      │
  [Tokeniser]  — splits text into tokens (subwords)
      │
  [Token Embeddings]  — map tokens to dense vectors
      │
  [Positional Encoding]  — encode token position in sequence
      │
  [N × Transformer Blocks]
      ├── Multi-Head Self-Attention  — "which tokens matter to which other tokens?"
      ├── Feed-Forward Network       — non-linear transformation per token
      └── Layer Normalisation + Residual connections
      │
  [Language Model Head]  — project to vocabulary logits
      │
  [Softmax + Sampling]  — temperature, top-p, top-k sampling
      │
  OUTPUT TOKEN (repeat until EOS)
```

In [ ]:
# Simulate tokenisation (simplified BPE-style)
def simple_tokenise(text: str) -> list:
    """Very simplified tokeniser — splits on spaces and punctuation."""
    tokens = []
    for word in text.split():
        if len(word) <= 4:
            tokens.append(word)
        else:
            # Simulate subword splitting
            tokens.append(word[:4])
            tokens.append(word[4:])
    return tokens


def estimate_tokens(text: str) -> int:
    """Rough token count estimate: ~4 chars per token for English."""
    return max(1, len(text) // 4)


def context_window_check(text: str, model_context_tokens: int) -> dict:
    """Check if text fits within a model's context window."""
    tokens = estimate_tokens(text)
    fits   = tokens <= model_context_tokens
    pct    = tokens / model_context_tokens * 100
    return {
        'text_tokens'     : tokens,
        'context_window'  : model_context_tokens,
        'usage_pct'       : f'{pct:.1f}%',
        'fits'            : fits,
        'tokens_remaining': model_context_tokens - tokens
    }


# Tokenisation demo
sample_text = 'SQL injection is a critical web application vulnerability that allows attackers to manipulate database queries.'
print(f'Text: "{sample_text[:60]}..."')
print(f'Simple tokens : {simple_tokenise(sample_text)}')
print(f'Estimated tokens: ~{estimate_tokens(sample_text)}')

# Context window comparison
print('\n=== CONTEXT WINDOW CHECK ===\n')
MODELS_CONTEXT = [
    ('GPT-4o',             128_000),
    ('Claude 3.5 Sonnet',  200_000),
    ('Gemini 1.5 Pro',   1_000_000),
    ('Llama 3.1 8B',        128_000),
    ('GPT-3.5 Turbo',         4_096),
]
long_doc = 'This is a long document. ' * 2000  # ~50K chars
for model, ctx in MODELS_CONTEXT:
    result = context_window_check(long_doc, ctx)
    fits   = '✅ FITS' if result['fits'] else '❌ TOO LONG'
    print(f"  {model:<25} ctx={ctx:>10,} tokens  usage={result['usage_pct']:>8}  {fits}")

## 5. Frontier LLM Comparison

In [ ]:
# Simulated benchmark scores — based on publicly reported figures (as of 2025)
FRONTIER_MODELS = [
    {
        'model'          : 'GPT-4o',
        'provider'       : 'OpenAI',
        'params'         : '~200B (est)',
        'context_window' : 128_000,
        'mmlu'           : 88.7,
        'humaneval'      : 90.2,
        'safety_score'   : 87,
        'cost_input_1m'  : 2.50,   # $ per 1M tokens
        'cost_output_1m' : 10.00,
        'multimodal'     : True,
        'open_source'    : False
    },
    {
        'model'          : 'Claude 3.5 Sonnet',
        'provider'       : 'Anthropic',
        'params'         : 'Unknown',
        'context_window' : 200_000,
        'mmlu'           : 88.3,
        'humaneval'      : 92.0,
        'safety_score'   : 93,
        'cost_input_1m'  : 3.00,
        'cost_output_1m' : 15.00,
        'multimodal'     : True,
        'open_source'    : False
    },
    {
        'model'          : 'Gemini 1.5 Pro',
        'provider'       : 'Google DeepMind',
        'params'         : 'Unknown',
        'context_window' : 1_000_000,
        'mmlu'           : 85.9,
        'humaneval'      : 84.1,
        'safety_score'   : 85,
        'cost_input_1m'  : 1.25,
        'cost_output_1m' : 5.00,
        'multimodal'     : True,
        'open_source'    : False
    },
    {
        'model'          : 'Llama 3.1 70B',
        'provider'       : 'Meta',
        'params'         : '70B',
        'context_window' : 128_000,
        'mmlu'           : 82.6,
        'humaneval'      : 80.5,
        'safety_score'   : 78,
        'cost_input_1m'  : 0.59,
        'cost_output_1m' : 0.79,
        'multimodal'     : False,
        'open_source'    : True
    },
    {
        'model'          : 'Mistral Large 2',
        'provider'       : 'Mistral AI',
        'params'         : '123B',
        'context_window' : 128_000,
        'mmlu'           : 84.0,
        'humaneval'      : 92.1,
        'safety_score'   : 80,
        'cost_input_1m'  : 2.00,
        'cost_output_1m' : 6.00,
        'multimodal'     : False,
        'open_source'    : False
    },
    {
        'model'          : 'DeepSeek-V3',
        'provider'       : 'DeepSeek',
        'params'         : '671B (MoE)',
        'context_window' : 128_000,
        'mmlu'           : 88.5,
        'humaneval'      : 91.6,
        'safety_score'   : 72,
        'cost_input_1m'  : 0.27,
        'cost_output_1m' : 1.10,
        'multimodal'     : False,
        'open_source'    : True
    },
]

def overall_model_score(m: dict) -> float:
    """Composite score: 40% capability, 30% safety, 30% cost efficiency."""
    capability   = (m['mmlu'] + m['humaneval']) / 2 / 100  # normalise to 0-1
    safety       = m['safety_score'] / 100
    # Cost score: lower cost = higher score (inverse, normalised)
    cost_score   = 1 - min(1.0, (m['cost_input_1m'] + m['cost_output_1m']) / 30)
    return round((0.40 * capability + 0.30 * safety + 0.30 * cost_score) * 100, 1)

for m in FRONTIER_MODELS:
    m['composite_score'] = overall_model_score(m)

ranked = sorted(FRONTIER_MODELS, key=lambda x: x['composite_score'], reverse=True)

print('=== FRONTIER LLM COMPARISON ===\n')
print(f'{"Rank":<5} {"Model":<22} {"Provider":<18} {"MMLU":<7} {"HumanEval":<12} {"Safety":<9} {"$/1M in":<10} {"Score"}')
print('-' * 95)
for i, m in enumerate(ranked, 1):
    oss   = '🔓' if m['open_source'] else '🔒'
    multi = '👁' if m['multimodal'] else '  '
    print(f"  {i:<4} {m['model']:<22} {m['provider']:<18} {m['mmlu']:<7} {m['humaneval']:<12} {m['safety_score']:<9} ${m['cost_input_1m']:<9} {m['composite_score']}  {oss}{multi}")
print('\n🔓 = Open source  🔒 = Closed  👁 = Multimodal')

## 6. Model Adaptation Strategies

In [ ]:
ADAPTATION_STRATEGIES = {
    'Prompt Engineering': {
        'description'  : 'Guide model behaviour through carefully crafted prompts — no training',
        'when_to_use'  : 'Quick iteration; API-only access; low-cost experimentation',
        'techniques'   : ['Zero-shot', 'Few-shot (examples in prompt)', 'Chain-of-thought', 'Role prompting', 'System prompt design'],
        'cost'         : 'Very low — just API token cost',
        'data_needed'  : 'None',
        'limitations'  : 'Context window bound; prompt injection risk; inconsistent across providers'
    },
    'RAG (Retrieval-Augmented Generation)': {
        'description'  : 'Retrieve relevant documents from a knowledge base and inject into prompt',
        'when_to_use'  : 'Domain knowledge that changes frequently; reduce hallucination; cite sources',
        'techniques'   : ['Dense retrieval (embeddings)', 'Sparse retrieval (BM25)', 'Hybrid search', 'Reranking'],
        'cost'         : 'Low-medium — embedding + vector DB + LLM API',
        'data_needed'  : 'Knowledge base documents (no labels needed)',
        'limitations'  : 'Retrieval quality limits generation quality; latency; chunk size tuning'
    },
    'Fine-tuning': {
        'description'  : 'Continue training a pre-trained model on domain-specific labelled data',
        'when_to_use'  : 'Consistent tone/style; specialised domain; low latency needed',
        'techniques'   : ['Full fine-tuning', 'LoRA (Low-Rank Adaptation)', 'QLoRA', 'RLHF', 'DPO'],
        'cost'         : 'Medium-high — GPU compute + labelled dataset preparation',
        'data_needed'  : '500–50,000+ labelled examples',
        'limitations'  : 'Risk of catastrophic forgetting; overfitting; requires ML expertise'
    },
    'Pre-training from Scratch': {
        'description'  : 'Train a model entirely from scratch on domain corpus',
        'when_to_use'  : 'Highly specialised domain (genomics, legal); data privacy; proprietary IP',
        'techniques'   : ['Causal LM pre-training', 'Masked LM (BERT-style)', 'MoE architecture'],
        'cost'         : 'Very high — millions of dollars in GPU compute',
        'data_needed'  : 'Billions of tokens of domain text',
        'limitations'  : 'Only feasible for large organisations; extremely costly'
    }
}

print('=== MODEL ADAPTATION STRATEGIES ===\n')
for strategy, details in ADAPTATION_STRATEGIES.items():
    print(f'  [{strategy}]')
    print(f'    Description  : {details["description"]}')
    print(f'    When to use  : {details["when_to_use"]}')
    print(f'    Cost         : {details["cost"]}')
    print(f'    Data needed  : {details["data_needed"]}')
    print(f'    Limitations  : {details["limitations"]}\n')

## 7. Model Selection Decision Framework

In [ ]:
def select_model(task_type: str, privacy_required: bool, budget: str,
                 needs_multimodal: bool, latency: str) -> dict:
    """
    Rule-based model selection guide.
    budget: 'low' | 'medium' | 'high'
    latency: 'real-time' | 'batch'
    """
    recommendations = []
    rationale       = []

    # Privacy-first → open-source self-hosted
    if privacy_required:
        recommendations += ['Llama 3.1 70B (self-hosted)', 'Mistral 7B (self-hosted)', 'DeepSeek-V3 (self-hosted)']
        rationale.append('Privacy required → self-hosted open-source model')
    elif budget == 'low':
        recommendations += ['Llama 3.1 8B (API)', 'Gemini 1.5 Flash', 'Mistral 7B (API)']
        rationale.append('Low budget → smaller, cheaper models')
    elif budget == 'high' and task_type in ('reasoning', 'code', 'complex_qa'):
        recommendations += ['GPT-4o', 'Claude 3.5 Sonnet', 'Gemini 1.5 Pro']
        rationale.append('High budget + complex task → frontier model')
    else:
        recommendations += ['GPT-4o mini', 'Claude 3 Haiku', 'Gemini 1.5 Flash']
        rationale.append('Medium budget → efficient mid-tier model')

    if needs_multimodal:
        recommendations = [r for r in recommendations if any(
            mm in r for mm in ['GPT-4o', 'Claude 3', 'Gemini'])]
        if not recommendations:
            recommendations = ['GPT-4o', 'Claude 3.5 Sonnet', 'Gemini 1.5 Pro']
        rationale.append('Multimodal required → vision-capable model')

    if latency == 'real-time':
        rationale.append('Real-time latency → avoid very large models; consider streaming')

    adaptation = 'RAG' if task_type == 'knowledge_qa' else \
                 'Fine-tuning' if task_type == 'domain_specific' else \
                 'Prompt engineering'

    return {
        'recommended_models': recommendations[:3],
        'adaptation_strategy': adaptation,
        'rationale'         : rationale
    }


# Test different scenarios
SCENARIOS = [
    {'label': 'Cybersecurity chatbot (privacy-sensitive)',  'args': ('knowledge_qa',    True,  'medium', False, 'real-time')},
    {'label': 'Code review assistant (internal tool)',     'args': ('code',             False, 'high',   False, 'batch')},
    {'label': 'Document analysis (images + text)',        'args': ('complex_qa',        False, 'high',   True,  'batch')},
    {'label': 'High-volume summarisation (low cost)',     'args': ('summarisation',     False, 'low',    False, 'batch')},
    {'label': 'Medical records (strict privacy)',          'args': ('domain_specific',   True,  'high',   False, 'batch')},
]

print('=== MODEL SELECTION GUIDE ===\n')
for s in SCENARIOS:
    rec = select_model(*s['args'])
    print(f"  Scenario: {s['label']}")
    print(f"    Recommended : {', '.join(rec['recommended_models'])}")
    print(f"    Adaptation  : {rec['adaptation_strategy']}")
    print(f"    Rationale   : {' | '.join(rec['rationale'])}\n")

## 8. AI Model Scorecard

In [ ]:
# Evaluate top 3 models against a cybersecurity use case
USE_CASE = 'Cybersecurity threat intelligence analyst assistant'

EVALUATION_CRITERIA = [
    ('Domain knowledge (cyber)',    0.25),
    ('Reasoning & analysis',        0.20),
    ('Safety & refusal accuracy',   0.20),
    ('Context window (long docs)',  0.15),
    ('Cost efficiency',             0.10),
    ('Privacy / deployment option', 0.10),
]

CANDIDATE_SCORES = [
    {'model': 'Claude 3.5 Sonnet', 'scores': [90, 92, 93, 85, 60, 70]},
    {'model': 'GPT-4o',            'scores': [91, 90, 87, 80, 65, 65]},
    {'model': 'Llama 3.1 70B',     'scores': [80, 82, 78, 80, 95, 98]},
]

print(f'=== MODEL SCORECARD — {USE_CASE} ===\n')
print(f'{"Criterion":<32} {"Weight":<8}', end='')
for c in CANDIDATE_SCORES:
    print(f'{c["model"]:<22}', end='')
print()
print('-' * 100)

weighted_totals = [0.0] * len(CANDIDATE_SCORES)
for i, (criterion, weight) in enumerate(EVALUATION_CRITERIA):
    print(f'  {criterion:<30} {weight:<8.0%}', end='')
    for j, c in enumerate(CANDIDATE_SCORES):
        score = c['scores'][i]
        weighted_totals[j] += score * weight
        print(f'{score:<22}', end='')
    print()

print('-' * 100)
print(f'  {"WEIGHTED TOTAL":<38}', end='')
for j, c in enumerate(CANDIDATE_SCORES):
    print(f'{weighted_totals[j]:.1f}{"":17}', end='')
print()

best = CANDIDATE_SCORES[weighted_totals.index(max(weighted_totals))]
print(f"\n  ✅ Recommended for this use case: {best['model']} (score: {max(weighted_totals):.1f}/100)")

## 9. Key Takeaways

| Concept | Takeaway |
|---------|----------|
| **Model selection** | Match model to task — don't default to GPT-4o for every use case |
| **Cost vs capability** | Open-source models (Llama, Mistral) can match frontier models at 10x lower cost |
| **Context window** | Long-context models (Gemini 1M) enable processing entire codebases or legal documents |
| **Adaptation hierarchy** | Start with prompt engineering → RAG → fine-tuning (in that order of cost/complexity) |
| **Safety matters** | Higher MMLU doesn't always mean safer — evaluate safety separately |
| **Privacy** | If data is sensitive, self-hosted open-source is the only responsible option |

### AI Model Quick Reference
| Need | Model Family | Example |
|------|-------------|--------|
| Best overall quality | Frontier LLM | Claude 3.5 Sonnet, GPT-4o |
| Lowest cost | Small open-source | Llama 3.1 8B, Mistral 7B |
| Privacy / on-prem | Open-source | Llama 3.1 70B, DeepSeek-V3 |
| Images + text | Multimodal | GPT-4o, Gemini 1.5 Pro |
| Code generation | Code LLM | Claude 3.5, DeepSeek Coder |
| Long documents | Large context | Gemini 1.5 Pro (1M tokens) |
| Semantic search | Embedding | text-embedding-3-large |
| Tabular / fraud | Classical ML | XGBoost, LightGBM |

---
*Part of the AI Series — Notebooks 14–21*